# 03 · Preprocessing for CNNs: spectrograms

<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/refactor/notebooks/03_preprocessing_dl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CNNs take the log spectrogram of the 8 independent leads. The STFT runs on the signals at their original 400 Hz after a 0.5 Hz high-pass, and only the bins up to 40 Hz are kept: the crop of the spectrum plays the role of the low-pass filter and of the resampling of the feature pipeline. The cohort and the folds come from `02_preprocessing_ml.ipynb`.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/chagas")
    if ROOT.exists():
        !git -C {ROOT} pull -q
    else:
        !git clone -q -b refactor https://github.com/dokunoale/chagas.git {ROOT}
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT))

In [ ]:
import warnings
from math import ceil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from src import leakage
from src import preprocessing as pp
from src import spectrogram as sg
from src.dataset import download, sources

DATA_DIR = Path("/content/data") if IN_COLAB else ROOT / "data"
CODE15_PARTS = [0]  # one part: ~20k exams, 2.7 GB zipped
N_PER_GROUP = 500   # records per source and class for the shortcut probe
SEED = 42

# Shared Drive folders (link or id) holding the original files; None downloads from Zenodo
CODE15_DRIVE = None
SAMITROP_DRIVE = None

# Processed arrays go to Drive on Colab, so the next notebooks can reuse them
SAVE_TO_DRIVE = IN_COLAB
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROCESSED_DIR = Path("/content/drive/MyDrive/chagas-data/processed")
else:
    PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ["CODE-15% neg", "CODE-15% pos", "SaMi-Trop pos"]
PALETTE = dict(zip(GROUPS, sns.color_palette("colorblind", len(GROUPS))))

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

## Data

Same raw files as in `01_dataset_analysis.ipynb`; downloads are skipped when the files already exist.

In [ ]:
download.download_code15(DATA_DIR, CODE15_PARTS, gdown=CODE15_DRIVE)
download.download_samitrop(DATA_DIR, gdown=SAMITROP_DRIVE)
meta = sources.load_all(DATA_DIR, CODE15_PARTS)

## 1. Cohort

Loaded from notebook 02, or rebuilt with the same seed if it is missing. Signal locations are refreshed from the local data folder.

In [ ]:
cohort_file = PROCESSED_DIR / "cohort.csv"
if cohort_file.exists():
    cohort = pp.load_cohort(cohort_file)
    located = meta.set_index("record_id").loc[cohort.record_id]
    cohort["file"], cohort["index"] = located.file.to_numpy(), located["index"].to_numpy()
else:
    cohort = pp.build_cohort(meta, seed=SEED)
    cohort["group"] = cohort.source + np.where(cohort.chagas, " pos", " neg")
    cohort.to_csv(cohort_file, index=False)
kept = cohort[cohort.array_index >= 0].sort_values("array_index").reset_index(drop=True)
kept.group.value_counts().reindex(GROUPS)

## 2. Time–frequency trade-off

A Hann window of length *w* resolves frequencies about 1/*w* apart and smears events shorter than *w* in time. The QRS lasts about 100 ms, so short windows keep the beat structure while long ones give finer frequency bins. Below, lead V1 of a CODE-15% positive with RBBB, with windows of 0.25, 0.5 and 1 s (1 Hz grid, 50 ms hop for display). The arrays use a 0.5 s window with a 0.1 s hop.

In [ ]:
example = kept[(kept.source == "CODE-15%") & kept.chagas & kept.RBBB.astype(str).eq("True")].sample(1, random_state=SEED)
x = sg.window(sources.read_signals(example)[0])
v1 = pp.LEADS.index("V1")

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
axes[0].plot(np.arange(len(x)) / sources.FS, x[:, v1], lw=0.8)
axes[0].set(title="V1, high-passed", ylabel="mV")
for ax, win in zip(axes[1:], [0.25, 0.5, 1.0]):
    s, f, t = sg.log_spectrogram(x, sources.FS, win_s=win, hop_s=0.05)
    ax.pcolormesh(t, f, s[v1], shading="nearest", cmap="magma")
    ax.set(title=f"window {win} s (resolution ~{1 / win:.0f} Hz)", ylabel="Hz")
axes[-1].set_xlabel("s")
plt.tight_layout()

## 3. Crop of the spectrum

Full-band spectrograms of lead II up to 200 Hz, with the 40 Hz limit in white. Above it the power is orders of magnitude lower and depends on the acquisition filter of each source (notebook 01, section 7).

In [ ]:
fig, axes = plt.subplots(1, len(GROUPS), figsize=(17, 4), sharey=True)
ii = pp.LEADS.index("II")
for ax, group in zip(axes, GROUPS):
    x = sg.window(sources.read_signals(kept[kept.group == group].sample(1, random_state=SEED))[0])
    s, f, t = sg.log_spectrogram(x, sources.FS, band=(0, 200))
    im = ax.pcolormesh(t, f, s[ii], shading="nearest", cmap="magma", vmin=-8, vmax=0)
    ax.axhline(pp.BAND[1], color="w", ls="--", lw=1)
    ax.set(title=f"{group} · II", xlabel="s")
axes[0].set_ylabel("Hz")
fig.colorbar(im, ax=axes, label="log10 PSD (mV²/Hz)");

## 4. Spectrograms of all records

float16 of shape (records, 8 leads, 40 frequencies, 65 frames), 1–40 Hz on a 1 Hz grid, frames every 0.1 s, in the order of `array_index`. The frequency and time axes are saved next to the array.

In [ ]:
shape = sg.preprocess(sources.read_signals(kept.iloc[:1])[0]).shape
specs = np.empty((len(kept), *shape), dtype=np.float16)
pos = 0
for _, raw in tqdm(sources.iter_signals(kept, chunk_size=2000), total=ceil(len(kept) / 2000)):
    specs[pos:pos + len(raw)] = np.stack([sg.preprocess(x) for x in raw])
    pos += len(raw)
np.save(PROCESSED_DIR / "spectrograms_dl.npy", specs)

_, freqs, times = sg.log_spectrogram(np.zeros((int(pp.WINDOW_S * sources.FS), 1)), sources.FS)
np.savez(PROCESSED_DIR / "spectrograms_dl_axes.npz", freqs=freqs, times=times, leads=np.array(pp.LEADS))
specs.shape

## 5. Average spectral profile

The time axis carries no alignment across records, so the comparison uses the mean over frames: one lead × frequency profile per record. Left, CODE-15% negatives; then the difference between CODE-15% positives and negatives, and between SaMi-Trop and CODE-15% positives.

In [ ]:
profile = specs.mean(axis=-1, dtype=np.float32)
groups = kept.group.to_numpy()
mean = {group: profile[groups == group].mean(axis=0) for group in GROUPS}
panels = [("CODE-15% neg", mean["CODE-15% neg"], "magma"),
          ("CODE-15% pos − neg", mean["CODE-15% pos"] - mean["CODE-15% neg"], "RdBu_r"),
          ("SaMi-Trop pos − CODE-15% pos", mean["SaMi-Trop pos"] - mean["CODE-15% pos"], "RdBu_r")]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (title, m, cmap) in zip(axes, panels):
    lim = np.abs(m).max() if cmap == "RdBu_r" else None
    im = ax.pcolormesh(freqs, np.arange(len(pp.LEADS)), m, cmap=cmap, shading="nearest",
                       vmin=-lim if lim else None, vmax=lim)
    ax.set(title=title, xlabel="Hz", yticks=range(len(pp.LEADS)), yticklabels=pp.LEADS)
    fig.colorbar(im, ax=ax, label="log10 PSD" if cmap == "magma" else "difference (log10)")
plt.tight_layout()

## 6. Shortcut probe on spectral profiles

The probe of notebooks 01 and 02 on the 320 values of the spectral profile (8 leads × 40 frequencies), on the same kind of sample. This input carries morphology as well as acquisition, so the Chagas probe measures signal content too; the source probe among positives is the one to watch.

In [ ]:
sample = kept.sample(frac=1, random_state=SEED).groupby("group").head(N_PER_GROUP)
X = pd.DataFrame(profile[sample.index.to_numpy()].reshape(len(sample), -1), index=sample.index)
g = sample.group
probes = {
    "source | positives (CODE-15% vs SaMi-Trop)": (g != "CODE-15% neg", g == "SaMi-Trop pos"),
    "Chagas | CODE-15% only": (sample.source == "CODE-15%", sample.chagas),
}

pd.Series({name: roc_auc_score(target[mask], leakage.source_probe(X[mask], target[mask], sample.patient_id[mask], seed=SEED))
           for name, (mask, target) in probes.items()}, name="ROC AUC").round(3)

## 7. Saved outputs

In [ ]:
for path in sorted(PROCESSED_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1e6:,.1f} MB")